# PCA Cluster Explorer — Cities Dataset
Interactive visualization of residual-stream activations in PC space.
Hover over points to see the statement, city, and country.

In [126]:
import torch
import pandas as pd
import plotly.express as px

import numpy as np
import plotly.io as pio
from IPython.display import display, HTML
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler

# good default for classic notebook / many Jupyter setups
pio.renderers.default = "iframe"

import sys      
sys.path.insert(0, "/storage/project/r-aivanova7-0/shared/eyas/geometry_of_truth_replication")
from src.activations import load_acts

from src.pca import run_pca, cross_dataset_pca, plot_cross_dataset_pca 

In [127]:
# ── Configuration ─────────────────────────────────────────────────────────────
MODEL   = "llama-2-13b"
LAYER   = 15
DATASET_NAME = "cities"
PCA_DIR     = "/storage/home/hcoda1/7/eayesh3/scratch/geometry_of_truth/pca"
ACTS_DIR     = "/storage/home/hcoda1/7/eayesh3/scratch/geometry_of_truth/acts"
DATASET_CSV = f"../datasets/{DATASET_NAME}.csv"

In [129]:
acts = load_acts(MODEL, DATASET_NAME, LAYER, ACTS_DIR, False)
PCAResults = run_pca(acts,k=50)

In [132]:
# ── Load PCA result ───────────────────────────────────────────────────────────
pca_path = f"{PCA_DIR}/{MODEL}/{DATASET_NAME}/pca_layer{LAYER}.pt"
pca = torch.load(pca_path, weights_only=True)

proj = pca["projections"].numpy()   # [n_statements, n_pcs]
ev   = pca["explained_var_ratio"]

proj_raw = PCAResults.projections.numpy()
ev_raw   = PCAResults.explained_var_ratio

print(f"Projections shape: {proj.shape}")
print(f"PC1: {ev[0]:.2%}  PC2: {ev[1]:.2%}  PC3: {ev[2]:.2%}")
print(f"Projections RAW shape: {proj_raw.shape}")
print(f"RAW PC1: {ev_raw[0]:.2%}  RAW PC2: {ev_raw[1]:.2%}  RAW PC3: {ev_raw[2]:.2%}")

Projections shape: (1496, 10)
PC1: 45.40%  PC2: 8.38%  PC3: 5.69%
Projections RAW shape: (1496, 50)
RAW PC1: 45.40%  RAW PC2: 8.38%  RAW PC3: 5.69%


In [118]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATASET_CSV)
assert len(df) == proj.shape[0], f"Row count mismatch: {len(df)} vs {proj.shape[0]}"

df["PC1"]   = proj[:, 0]
df["PC2"]   = proj[:, 1]
df["PC3"]   = proj[:, 2]
df["PC4"]   = proj[:, 3]
df["PC5"]   = proj[:, 4]
df["PC6"]   = proj[:, 5]
df["PC7"]   = proj[:, 6]
df["PC8"]   = proj[:, 7]
df["PC9"]   = proj[:, 8]
df["PC10"]   = proj[:, 9]
df["truth"] = df["label"].map({1: "True", 0: "False"})

In [119]:
# ── 2D scatter ────────────────────────────────────────────────────────────────
fig = px.scatter(
    df,
    x="PC1", y="PC2",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "city": True, "country": True,
                "correct_country": True, "PC1": False, "PC2": False},
    title=f"{MODEL} — cities — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_traces(marker_size=5)
fig.show()

In [120]:
X = df[["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8"  ]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2)
X["cluster"] = kmeans.fit_predict(X)
X["PC1_scaled"] = X_scaled[:,0]
X["PC2_scaled"] = X_scaled[:,1]
X["PC3_scaled"] = X_scaled[:,2]
X["PC4_scaled"] = X_scaled[:,3]
X["PC5_scaled"] = X_scaled[:,4]
X["PC6_scaled"] = X_scaled[:,5]
X["PC7_scaled"] = X_scaled[:,6]
X["PC8_scaled"] = X_scaled[:,7]
print(np.sum(X["cluster"]==1))
outlier_mask = X["cluster"]==1

40


In [122]:
# ── 3D scatter ────────────────────────────────────────────────────────────────
fig3d = px.scatter_3d(
    df[~outlier_mask],
    x="PC1", y="PC2", z="PC3",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "city": True, "country": True,
                "correct_country": True},
    title=f"{MODEL} — cities — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}, PC3: {ev[2]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig3d.update_traces(marker_size=3)
fig3d.show()

In [133]:
PCAResults_adjusted = run_pca(acts[~outlier_mask],k=50)
proj = PCAResults_adjusted.projections.numpy()
ev   = PCAResults_adjusted.explained_var_ratio
print(f"Projections shape: {proj.shape}")
print(f"PC1: {ev[0]:.2%}  PC2: {ev[1]:.2%}  PC3: {ev[2]:.2%}")

Projections shape: (1456, 50)
PC1: 23.48%  PC2: 9.23%  PC3: 5.40%


In [136]:
# ── Load dataset ──────────────────────────────────────────────────────────────
df = pd.read_csv(DATASET_CSV)
df = df[~outlier_mask]
assert len(df) == proj.shape[0], f"Row count mismatch: {len(df)} vs {proj.shape[0]}"

df["PC1"]   = proj[:, 0]
df["PC2"]   = proj[:, 1]
df["PC3"]   = proj[:, 2]
df["PC4"]   = proj[:, 3]
df["PC5"]   = proj[:, 4]
df["PC6"]   = proj[:, 5]
df["PC7"]   = proj[:, 6]
df["PC8"]   = proj[:, 7]
df["PC9"]   = proj[:, 8]
df["PC10"]   = proj[:, 9]
df["truth"] = df["label"].map({1: "True", 0: "False"})

In [137]:
# ── 2D scatter ────────────────────────────────────────────────────────────────
fig = px.scatter(
    df,
    x="PC1", y="PC2",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "city": True, "country": True,
                "correct_country": True, "PC1": False, "PC2": False},
    title=f"{MODEL} — cities — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_traces(marker_size=5)
fig.show()

In [51]:
# ── ACTS Configuration ─────────────────────────────────────────────────────────────
ACTS_DATASET_NAME = "neg_cities"
ACTS_DIR     = "/storage/home/hcoda1/7/eayesh3/scratch/geometry_of_truth/acts"
ACTS_DATASET_CSV = f"../datasets/{ACTS_DATASET_NAME}.csv"

acts_path = f"{ACTS_DIR}/{MODEL}/{ACTS_DATASET_NAME}/pca_layer{LAYER}.pt"
acts = load_acts(MODEL, ACTS_DATASET_NAME, LAYER, ACTS_DIR, False)


torch.Size([1496, 5120])

In [82]:
source_result, tgt_proj, src_labels, tgt_labels = cross_dataset_pca(                
      source_dataset="cities",                                        
      target_dataset="neg_cities",                                                   
      model_name="llama-2-13b",                                                       
      layer=15,                                                                    
      acts_dir="/storage/home/hcoda1/7/eayesh3/scratch/geometry_of_truth/acts",
      k=10
  )     

In [83]:
fig = plot_cross_dataset_pca(                                                     
      source_result, tgt_proj, src_labels, tgt_labels,                              
      source_name="cities", target_name="sp_en_trans",
      title="sp_en_trans projected onto cities PC space — layer 16",
  )
fig.show()

In [91]:
joint_proj.shape

(2992, 10)

In [100]:
# ── Load dataset ──────────────────────────────────────────────────────────────
city_df = pd.read_csv(DATASET_CSV)
neg_city_df = pd.read_csv(ACTS_DATASET_CSV)

city_df["dataset_source"] = "city"
neg_city_df["dataset_source"] = "neg_city"

df = pd.concat([city_df,neg_city_df],ignore_index=True, join="outer")

joint_proj = np.concat([proj,tgt_proj.numpy()],axis=0)
assert len(df) == joint_proj.shape[0], f"Row count mismatch: {len(df)} vs {proj.shape[0]}"

df["PC1"]   = joint_proj[:, 0]
df["PC2"]   = joint_proj[:, 1]
df["PC3"]   = joint_proj[:, 2]
df["PC4"]   = joint_proj[:, 3]
df["PC5"]   = joint_proj[:, 4]
df["PC6"]   = joint_proj[:, 5]
df["PC7"]   = joint_proj[:, 6]
df["PC8"]   = joint_proj[:, 7]
df["PC9"]   = joint_proj[:, 8]
df["PC10"]   = joint_proj[:, 9]
df["truth"] = df["label"].map({1: "True", 0: "False"})

In [105]:
# ── 2D scatter ────────────────────────────────────────────────────────────────
fig = px.scatter(
    df,
    x="PC3", y="PC2",
    symbol="dataset_source",
    symbol_map={
        "city": "circle",
        "neg_city": "cross"
    },
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "city": True, "country": True,
                "correct_country": True, "PC1": False, "PC2": False},
    title=f"{MODEL} — cities — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_traces(marker_size=5)
fig.show()

In [115]:
X = df[["PC1", "PC2", "PC3", "PC4", "PC5", "PC6", "PC7", "PC8"  ]]

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

kmeans = KMeans(n_clusters=2)
X["cluster"] = kmeans.fit_predict(X)
X["PC1_scaled"] = X_scaled[:,0]
X["PC2_scaled"] = X_scaled[:,1]
X["PC3_scaled"] = X_scaled[:,2]
X["PC4_scaled"] = X_scaled[:,3]
X["PC5_scaled"] = X_scaled[:,4]
X["PC6_scaled"] = X_scaled[:,5]
X["PC7_scaled"] = X_scaled[:,6]
X["PC8_scaled"] = X_scaled[:,7]
print(np.sum(X["cluster"]==1))
outlier_mask = X["cluster"]==1

80


In [114]:
# ── 2D scatter ────────────────────────────────────────────────────────────────
fig = px.scatter(
    X[X["cluster"]==0],
    x="PC1_scaled", y="PC2_scaled",
    color="cluster",
    color_discrete_map={"1": "#d62728", "0": "#1f77b4"},
    title=f"{MODEL} — cities — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig.update_yaxes(scaleanchor="x", scaleratio=1)
fig.update_traces(marker_size=5)
fig.show()

In [117]:
# ── 3D scatter ────────────────────────────────────────────────────────────────
fig3d = px.scatter_3d(
    df[~outlier_mask],
    x="PC1", y="PC2", z="PC3",
    symbol="dataset_source",
    symbol_map={
        "city": "circle",
        "neg_city": "cross"
    },
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={"statement": True, "city": True, "country": True,
                "correct_country": True},
    title=f"{MODEL} — cities — layer {LAYER}  |  PC1: {ev[0]:.1%}, PC2: {ev[1]:.1%}, PC3: {ev[2]:.1%}",
    template="plotly_white",
    opacity=0.7,
)
fig3d.update_traces(marker_size=3)
fig3d.show()

In [9]:
# ── Scree plot — explained variance per PC ────────────────────────────────────
n_show = min(10, len(ev))
scree_df = pd.DataFrame({
    "PC": [f"PC{i+1}" for i in range(n_show)],
    "Explained variance": ev[:n_show],
})
px.bar(
    scree_df, x="PC", y="Explained variance",
    title=f"Scree plot — {MODEL} — cities — layer {LAYER}",
    template="plotly_white",
).show()

In [10]:
#%pip install umap-learn

In [11]:
# ── UMAP exploration with configurable settings ───────────────────────────────
import umap

UMAP_NEIGHBORS = 30
UMAP_MIN_DIST  = 1
UMAP_METRIC    = "euclidean"

umap_input = proj

umap_2d = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    n_components=2,
    metric=UMAP_METRIC,
    random_state=42,
)

umap_proj_2d = umap_2d.fit_transform(umap_input)

umap_df_2d = df.copy()
umap_df_2d["UMAP1"] = umap_proj_2d[:, 0]
umap_df_2d["UMAP2"] = umap_proj_2d[:, 1]



/storage/project/r-aivanova7-0/eayesh3/conda_envs/EM_env/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(


In [12]:
fig_umap = px.scatter(
    umap_df_2d,
    x="UMAP1", y="UMAP2",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={
        "statement": True,
        "city": True,
        "country": True,
        "correct_country": True,
        "UMAP1": False,
        "UMAP2": False,
    },
    title=(
        f"UMAP 2D — {MODEL} — cities — layer {LAYER}"
        f" | n_neighbors={UMAP_NEIGHBORS}, min_dist={UMAP_MIN_DIST}, metric={UMAP_METRIC}"
    ),
    template="plotly_white",
    opacity=0.7,
)
fig_umap.update_yaxes(scaleanchor="x", scaleratio=1)
fig_umap.update_traces(marker_size=5)
fig_umap.show()



In [13]:
umap_3d = umap.UMAP(
    n_neighbors=UMAP_NEIGHBORS,
    min_dist=UMAP_MIN_DIST,
    n_components=3,
    metric=UMAP_METRIC,
    random_state=42,
)

umap_proj_3d = umap_3d.fit_transform(umap_input)

umap_df_3d = df.copy()
umap_df_3d["UMAP1"] = umap_proj_3d[:, 0]
umap_df_3d["UMAP2"] = umap_proj_3d[:, 1]
umap_df_3d["UMAP3"] = umap_proj_3d[:, 2]

fig_umap_3d = px.scatter_3d(
    umap_df_3d,
    x="UMAP1", y="UMAP2", z="UMAP3",
    color="truth",
    color_discrete_map={"True": "#d62728", "False": "#1f77b4"},
    hover_data={
        "statement": True,
        "city": True,
        "country": True,
        "correct_country": True,
    },
    title=(
        f"UMAP 3D — {MODEL} — cities — layer {LAYER}"
        f" | n_neighbors={UMAP_NEIGHBORS}, min_dist={UMAP_MIN_DIST}, metric={UMAP_METRIC}"
    ),
    template="plotly_white",
    opacity=0.7,
)
fig_umap_3d.update_traces(marker_size=3)
fig_umap_3d.show()

/storage/project/r-aivanova7-0/eayesh3/conda_envs/EM_env/lib/python3.14/site-packages/umap/umap_.py:1952: UserWarning: n_jobs value 1 overridden to 1 by setting random_state. Use no seed for parallelism.
  warn(
